## Import Modules

In [ ]:
import pandas as pd
import numpy as np
import spacy
import textstat
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from bert_score import score
from rouge_score import rouge_scorer
import nltk
from nltk.util import ngrams
import os
from openai import OpenAI

nltk.download('punkt')

nlp = spacy.load("en_core_web_sm")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

## Evaluation Function

In [ ]:
class ContextAwareEvaluator:

    def __init__(self, query, output):
        self.query = query
        self.output = output

    ########################################
    # Semantic Similarity
    ########################################

    def semantic_similarity(self, query, response):
        q_emb = embedding_model.encode([query])
        r_emb = embedding_model.encode([response])

        sim = cosine_similarity(q_emb, r_emb)[0][0]
        return float(sim)

    ########################################
    # BERTScore
    ########################################

    def bertscore(self, query, response):
        P, R, F1 = score([response], [query], lang='en', verbose=False)
        return float(F1[0])

    ########################################
    # ROUGE
    ########################################

    def rouge_score_eval(self, query, response):
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
        scores = scorer.score(query, response)

        return {
            'rouge1': scores['rouge1'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        }

    ########################################
    # Entity Overlap
    ########################################

    def entity_overlap(self, query, response):
        q_doc = nlp(query)
        r_doc = nlp(response)

        q_entities = set([ent.text.lower() for ent in q_doc.ents])
        r_entities = set([ent.text.lower() for ent in r_doc.ents])

        if len(q_entities) == 0:
            return 0

        overlap = len(q_entities.intersection(r_entities)) / len(q_entities)
        return overlap

    ########################################
    # Keyword Coverage
    ########################################

    def keyword_coverage(self, query, response):
        vectorizer = TfidfVectorizer(stop_words='english')

        tfidf = vectorizer.fit_transform([query, response])

        query_words = set(query.lower().split())
        response_words = set(response.lower().split())

        if len(query_words) == 0:
            return 0

        return len(query_words.intersection(response_words)) / len(query_words)

    ########################################
    # Specificity Score
    ########################################

    def specificity_score(self, response):
        doc = nlp(response)

        named_entities = len(doc.ents)
        noun_chunks = len(list(doc.noun_chunks))
        unique_words = len(set(response.split()))
        total_words = len(response.split())

        if total_words == 0:
            return 0

        specificity = (
            (named_entities * 0.4) +
            (noun_chunks * 0.3) +
            ((unique_words / total_words) * 0.3)
        )

        return specificity

    ########################################
    # Diversity Metrics
    ########################################

    def distinct_n(self, text, n=2):
        tokens = nltk.word_tokenize(text.lower())

        if len(tokens) < n:
            return 0

        ng = list(ngrams(tokens, n))

        return len(set(ng)) / len(ng)

    ########################################
    # Readability
    ########################################

    def readability(self, text):
        return textstat.flesch_reading_ease(text)

    ########################################
    # Final Score
    ########################################

    def final_score(self, metrics):
        return (
            0.30 * metrics['semantic_similarity'] +
            0.20 * metrics['bertscore'] +
            0.15 * metrics['entity_overlap'] +
            0.15 * metrics['specificity'] +
            0.10 * metrics['distinct_2'] +
            0.10 * metrics['readability_normalized']
        )

    ########################################
    # Evaluate Single Response
    ########################################

    def evaluate_response(self, query, response):

        semantic_sim = self.semantic_similarity(query, response)
        bert_f1 = self.bertscore(query, response)

        rouge = self.rouge_score_eval(query, response)

        entity_overlap = self.entity_overlap(query, response)
        keyword_cov = self.keyword_coverage(query, response)

        specificity = self.specificity_score(response)

        distinct_1 = self.distinct_n(response, 1)
        distinct_2 = self.distinct_n(response, 2)

        readability = self.readability(response)
        readability_normalized = min(max(readability / 100, 0), 1)

        metrics = {
            'semantic_similarity': semantic_sim,
            'bertscore': bert_f1,
            'rouge1': rouge['rouge1'],
            'rougeL': rouge['rougeL'],
            'entity_overlap': entity_overlap,
            'keyword_coverage': keyword_cov,
            'specificity': specificity,
            'distinct_1': distinct_1,
            'distinct_2': distinct_2,
            'readability': readability,
            'readability_normalized': readability_normalized
        }

        metrics['final_score'] = self.final_score(metrics)

        return metrics

    ########################################
    # Full Dataset Evaluation
    ########################################

    def run(self):

        metrics = self.evaluate_response(self.query, self.output)
        
        return metrics




SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2196-2197: truncated \uXXXX escape (1256036624.py, line 346)

## Results

In [ ]:
if __name__ == '__main__':

    query = 'Difference between deep learning and machine learning?'
    output = '''
Learning Path **Title:** *Understanding Generalization Failure for Better Model Design*
Goal: By the end of this path you will be able to explain why mathematical (statistical / machine‑learning) models sometimes fail to generalize and use that understanding to make smarter design choices.

Stage 1 – Foundations of Generalization
1️⃣ What is generalization?
Definition: In machine learning, generalization is the ability of a trained model to make accurate predictions on new, unseen data that comes from the same underlying process as the training data.
Why it matters: A model that only memorizes the training set may achieve a perfect training loss, but it is useless in the real world where we must predict on fresh examples.
1️⃣2️⃣ Training vs. Test Performance
Metric	Computed on	What it tells you
Training error	Data the model has already seen	How well the model fits the known data (often optimistic).
Test (or validation) error	Fresh, held‑out data	How well the model generalizes to new situations.
If the test error is much higher than the training error, the model is over‑fitting – a classic sign of poor generalization.

📚 Worked Example
Suppose you fit a 5‑degree polynomial to 30 points sampled from a noisy sine wave.

Dataset	Mean Squared Error
Training	0.02 (very low)
Test (new points)	1.45 (high)
The model captured the noise in the training set, so it fails to predict the true sine shape on new data → generalization failure.

🤔 Reflection / Mini‑Activity
Think‑pair (or write) for 2 minutes:
If you were a hiring manager and received two models – one with 0 % training error and 30 % test error, the other with 10 % training error and 12 % test error – which would you choose and why?

Stage 2 – Bias–Variance Trade‑off and Model Complexity
2️⃣ Bias–Variance Decomposition (conceptual)
The expected prediction error for a point x can be split into three parts:

[ \underbrace{\text{Bias}^2(x)}{\text{Systematic error}} ;+; \underbrace{\text{Variance}(x)}{\text{Sensitivity to data}} ;+; \underbrace{\sigma^2}_{\text{Irreducible noise}} ]

Bias → error from overly simplistic assumptions (e.g., fitting a straight line to a curved relationship).
Variance → error from the model changing dramatically when the training data changes (highly flexible models).
2️⃣2️⃣ Model Complexity ↔ Bias & Variance
Model Complexity	Bias	Variance
Low (e.g., linear regression)	High (under‑fit)	Low
Medium (e.g., shallow decision tree)	Moderate	Moderate
High (e.g., deep neural net)	Low (can fit almost anything)	High (prone to over‑fit)
2️⃣3️⃣ Learning Curves – Visual Diagnosis
Curve	Interpretation
Training error ↓, Test error ↓, then plateau together	Good capacity; model generalizes.
Training error ↓, Test error ↓ then ↑	Over‑fitting: model memorizes training data.
Both errors stay high	Under‑fitting: model too simple.
📚 Worked Example (Learning Curves)
Train a decision‑tree classifier on the Iris dataset with varying depths:

Max depth	Training accuracy	Test accuracy
1	0.67	0.66
5	0.97	0.94
20	1.00	0.78
Depth = 20 gives zero training error (low bias) but a big drop in test accuracy (high variance) → generalization failure.

🤔 Activity
Plot it yourself: Using any dataset you like, train models of increasing complexity (e.g., polynomial degree 1‑10). Record training & test MSE and draw the two curves. Identify where the “sweet spot” lies and write a short sentence explaining why that point balances bias and variance.

Stage 3 – Overfitting, Regularization, and Cross‑Validation
3️⃣ Spotting Overfitting
Symptom	How to detect
Large gap between training and validation loss	Plot learning curves.
High variance in performance across folds	Use cross‑validation.
Model parameters with huge magnitude (e.g., weights ≫ 1)	Inspect weight distribution.
3️⃣2️⃣ Regularization Techniques (intuition)
Technique	What it does	Typical hyper‑parameter
L1 (Lasso)	Adds (\lambda\sum	w_i
L2 (Ridge)	Adds (\lambda\sum w_i^2) → shrinks weights toward 0, smooths solution.	(\lambda)
Dropout (NN)	Randomly zeroes a fraction of neurons each iteration → prevents co‑adaptation.	Drop probability p
Early stopping	Stops training when validation loss stops improving → avoids fitting noise.	Patience (epochs)
3️⃣3️⃣ Cross‑Validation (CV) – Estimating Generalization
k‑fold CV: Split data into k equal parts. Train on k‑1 parts, validate on the remaining part; repeat k times.
Result: Average validation error ≈ expected test error.
Model selection: Choose hyper‑parameters (e.g., λ, depth) that minimize the CV error.
📚 Worked Example (L2 Regularization)
Fit ridge regression on the Boston housing dataset with three λ values:

λ	Training RMSE	CV RMSE
0 (no reg)	2.1	4.8
1	2.3	3.5
10	3.0	3.2
λ = 10 reduces variance (CV error drops) at the cost of a bit more bias (higher training error). The best trade‑off is λ = 10 → better generalization.

🤔 Activity
Mini‑experiment: Pick a dataset, train a linear model with L1, L2, and no regularization. Record training & validation errors. Which regularizer gave the smallest validation error? Write a short paragraph linking the result to bias‑variance intuition.

Stage 4 – Data‑Related Challenges: Distribution Shift & Assumption Violations
4️⃣ What is distribution shift?
Type	Definition	Example
Covariate shift	(P_{\text{train}}(X) \neq P_{\text{test}}(X)) while (P(Y	X)) stays the same.
Label shift	(P_{\text{train}}(Y) \neq P_{\text{test}}(Y)) but (P(X	Y)) unchanged.
Concept shift (a.k.a. concept drift)	The conditional distribution changes: (P_{\text{train}}(Y	X) \neq P_{\text{test}}(Y
4️⃣2️⃣ Violations of Model Assumptions
Assumption	Why it matters	Consequence when violated
Linearity (e.g., linear regression)	Model assumes outcome is a linear combination of features.	Non‑linear relationships cause systematic bias.
Independence of errors	Errors are assumed uncorrelated.	Autocorrelated residuals inflate variance estimates.
Homoskedasticity (constant variance)	Uniform noise level across X.	Heteroskedastic noise leads to inefficient estimates.
Feature independence (naïve Bayes)	Features are conditionally independent given the class.	Correlated features degrade probability estimates.
4️⃣3️⃣ Illustrative Scenarios
Medical imaging – Training images come from a high‑resolution scanner; test images are from a cheaper scanner (covariate shift). The model’s pixel‑level statistics change → performance drops.
Credit scoring – During a recession, default rates rise (label shift). A model trained on pre‑recession data under‑estimates risk.
Self‑driving cars – Weather conditions change from sunny (training) to snowy (test) → road texture distribution shifts → lane‑detection model fails.
📚 Worked Example (Covariate Shift)
You train a house‑price predictor on data from urban neighborhoods only. When you apply it to suburban houses, the feature distribution (e.g., lot size) is different. The model systematically under‑predicts prices because it never saw large lots during training → a classic covariate‑shift failure.

🤔 Activity
Data audit: Take any dataset you have (or a public one). Split it into two halves based on a natural attribute (e.g., time, geography). Compare basic statistics (means, variances) of each feature across the splits. Write down any noticeable differences – could these cause a shift if you trained on one half and tested on the other?

Stage 5 – Inductive Bias, Domain Adaptation, and the Curse of Dimensionality
5️⃣ Inductive Bias – “What we assume before seeing data”
Definition: The set of assumptions a learning algorithm uses to prefer one hypothesis over another when multiple fit the training data equally well.
Why it matters: With limited data, bias steers learning toward plausible solutions; too weak a bias → high variance, too strong → high bias.
Algorithm	Typical bias
k‑NN	Smoothness: nearby points have similar labels.
Linear models	Linearity: relationships are linear.
Decision trees	Axis‑aligned splits (features are considered independently).
Neural nets	Compositional hierarchy (local patterns combine into higher‑level features).
5️⃣2️⃣ Basic Domain Adaptation Strategies
Strategy	Idea	When useful
Instance re‑weighting	Give more weight to training samples that look like target data (e.g., importance weighting).	Covariate shift with known density ratios.
Feature alignment	Learn a transformation that makes source and target feature distributions similar (e.g., CORAL, adversarial domain adaptation).	Visual domain shift (different cameras).
Fine‑tuning	Start from a model trained on source data, then continue training on a small amount of target data.	Small labeled target set available.
5️⃣3️⃣ Curse of Dimensionality
Phenomenon: As the number of features d grows, the volume of the space grows exponentially, making data points sparse.
Impact on generalization:
Distance‑based methods (k‑NN, kernel SVM) become unreliable because all points appear equally far apart.
Models need exponentially more data to achieve the same level of confidence.
Simple Illustration
In a 1‑D unit interval, 10 uniformly random points give an average spacing of 0.1. In a 10‑D unit hypercube, 10 points leave most of the volume empty; the nearest neighbor distance is close to the diameter of the cube (√10 ≈ 3.16), so “nearness” loses meaning.

📚 Worked Example (Feature Selection)
You have a text‑classification problem with 20 000 word‑frequency features but only 500 labeled documents. A linear SVM without any regularization over‑fits dramatically (training accuracy ≈ 100 %, test ≈ 55 %). Adding L1 regularization (which forces many coefficients to zero) reduces the model to ~300 active words, and test accuracy rises to 78 % – the bias introduced by sparsity combats the curse of dimensionality.

🤔 Activity
Feature‑reduction brainstorm: List three ways you could reduce dimensionality for a high‑dimensional dataset (e.g., images, text). For each, note how it might improve generalization and what bias it introduces.

Stage 6 – Synthesizing Knowledge for Model Design
6️⃣ Diagnostic Checklist
Symptom	Likely Root Cause	Quick Fix
Large training‑test gap, smooth learning curves	Over‑fitting (high variance)	Add regularization, reduce model capacity, collect more data, use early stopping.
Both training & test errors high, flat curves	Under‑fitting (high bias)	Increase capacity, add non‑linear features, reduce regularization.
'''

    evaluator = ContextAwareEvaluator(query, output)

    results = evaluator.run()

    print("=" * 80)
    print("EVALUATION RESULTS")
    print("=" * 80)

    print(results)
    

## Raw LLM Content Generation

In [7]:
# Groq
api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


In [24]:
model1, model2 = "openai/gpt-oss-120b", "meta-llama/llama-4-scout-17b-16e-instruct"
query = 'Why mathematical Models fail to generalize?'
chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"""{query}""",
        }
    ],
    model=model2,
)

reply = chat_completion.choices[0].message.content.lower()

print(reply)

mathematical models can fail to generalize for several reasons. generalization refers to the ability of a model to perform well on unseen data, not just the data it was trained on. when a model fails to generalize, it means that it is not capturing the underlying patterns or relationships in a way that is applicable beyond the specific dataset it was trained on. here are some key reasons why mathematical models, particularly those in machine learning and data science, might fail to generalize:

1. **overfitting**: this is one of the most common reasons. overfitting occurs when a model is too complex and learns the noise and outliers in the training data rather than the underlying pattern. as a result, it performs well on the training data but poorly on new, unseen data.

2. **underfitting**: conversely, underfitting happens when a model is too simple to capture the underlying patterns in the data. such a model performs poorly on both the training data and new data.

3. **lack of repres